# Preprocessing Transactions

This file preprocesses the transaction data 

In [6]:
import pandas as pd
import numpy as np
import geopandas as gpd
import sys, os, glob
import re
import math
import matplotlib.pyplot as plt
from pyspark.sql import functions as F
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType, DateType, DoubleType
from datetime import datetime, timedelta

In [7]:
sys.path.insert(0, "../scripts")
from spark_setup import get_spark
spark = get_spark()

In [11]:
transactions = spark.read.parquet("../data/landing/transactions_20210828_20220227_snapshot")
transactions.show()

+-------+------------+------------------+--------------------+--------------+
|user_id|merchant_abn|      dollar_value|            order_id|order_datetime|
+-------+------------+------------------+--------------------+--------------+
|  14935| 79417999332|136.06570809815838|23acbb7b-cf98-458...|    2021-11-26|
|      1| 46451548968| 72.61581642788431|76bab304-fa2d-400...|    2021-11-26|
|  14936| 89518629617|3.0783487174439297|a2ae446a-2959-41c...|    2021-11-26|
|      1| 49167531725| 51.58228625503599|7080c274-17f7-4cc...|    2021-11-26|
|  14936| 31101120643|25.228114942417797|8e301c0f-06ab-45c...|    2021-11-26|
|      2| 67978471888| 691.5028234458998|0380e9ad-b0e8-420...|    2021-11-26|
|  14936| 60956456424|102.13952056640888|5ac3da9c-5147-452...|    2021-11-26|
|      2| 47644196714| 644.5220654863093|4e368e44-86f8-4de...|    2021-11-26|
|  14938| 39649557865|209.12780951421405|4d78cd01-4bab-494...|    2021-11-26|
|      3| 88402174457| 141.0387993699113|c50c957d-ecfc-430...|  

In [12]:
transactions.write.mode('overwrite').parquet('.././data/landing/transactions/total_transactions')

In [13]:
transactions = spark.read.parquet(".././data/landing/transactions/total_transactions")
transactions.show(truncate = False)

+-------+------------+------------------+------------------------------------+--------------+
|user_id|merchant_abn|dollar_value      |order_id                            |order_datetime|
+-------+------------+------------------+------------------------------------+--------------+
|14936  |32709545238 |440.47840693194706|a600de7a-6055-427b-9f0c-54ece8d0bfe3|2021-12-10    |
|1      |22953464223 |177.42297784724644|d8433f98-6046-4acf-9009-55cc0c032b0b|2021-12-10    |
|14937  |31585975447 |11.258260082442414|5e9dd2b5-abce-4912-b731-05cdc80a39f4|2021-12-10    |
|1      |65426342453 |646.6581742148818 |7dce3cc6-039c-4dd0-bb99-9fe894bbc920|2021-12-10    |
|14938  |89726005175 |85.71368390829582 |1caab391-d0d0-406e-a7d6-05bc43519020|2021-12-10    |
|2      |68216911708 |20.564746825448765|6bd788f0-5d76-4bba-8abd-ffa99f3c2603|2021-12-10    |
|14938  |49891706470 |47.34217367264105 |44897f0b-4f88-4f97-8b85-3af4847edfef|2021-12-10    |
|2      |14058755389 |383.4670079309222 |7d2748f7-3a20-4907-

In [14]:
transactions.printSchema()

root
 |-- user_id: long (nullable = true)
 |-- merchant_abn: long (nullable = true)
 |-- dollar_value: double (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_datetime: date (nullable = true)



Ensure all datatypes are consistent across tables.

In [15]:
transactions = transactions.withColumn('user_id', F.col('user_id').cast(StringType()))
transactions = transactions.withColumn('merchant_abn', F.col('merchant_abn').cast(StringType()))
transactions.printSchema()

root
 |-- user_id: string (nullable = true)
 |-- merchant_abn: string (nullable = true)
 |-- dollar_value: double (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_datetime: date (nullable = true)



In [21]:
def report_shape(df, label):
    n = df.count()
    print(f"{label}: {n:,} rows")
    return n

n0 = report_shape(transactions, "0. raw transactions")

# --- Business-rule filtering ---
# dollar_value must be strictly positive -- a $0 or negative transaction
# isn't a real purchase (refunds/corrections, if any, aren't modelled here).
step1 = transactions.filter(F.col("dollar_value") > 0)
n1 = report_shape(step1, "1. dollar_value > 0")

clean = step1
print(f"Retained: {n1:,} / {n0:,} ({n1/n0:.1%})")

0. raw transactions: 4,508,106 rows
1. dollar_value > 0: 4,508,106 rows
Retained: 4,508,106 / 4,508,106 (100.0%)


In [22]:
# --- Outlier removal: dollar_value (log-N IQR rule) ---
# Raw dollar_value is right-skewed (many small purchases, a long tail of
# large ones), so IQR is computed on log1p(dollar_value) rather than the
# raw scale. Per the log-N rule: flag values more than k x IQR from the
# box, where k = sqrt(log n) - 0.5 for n > 100.

clean = clean.withColumn("log_dollar_value", F.log1p(F.col("dollar_value")))

n_rows = clean.count()
k = np.sqrt(np.log(n_rows)) - 0.5

q1, q3 = clean.approxQuantile("log_dollar_value", [0.25, 0.75], 0.0)
iqr = q3 - q1

lower_bound = q1 - k * iqr
upper_bound = q3 + k * iqr

print(f"n = {n_rows:,}, k = {k:.4f}")
print(f"Log-scale Q1: {q1:.4f}, Q3: {q3:.4f}, IQR: {iqr:.4f}")
print(f"Log-scale bounds: [{lower_bound:.4f}, {upper_bound:.4f}]")

n_before_outlier = report_shape(clean, "2. before outlier removal")

curated = clean.filter(
    (F.col("log_dollar_value") >= lower_bound) &
    (F.col("log_dollar_value") <= upper_bound)
).drop("log_dollar_value")

n_after_outlier = report_shape(curated, "3. after log-N IQR outlier removal")
print(f"Removed as outliers: {n_before_outlier - n_after_outlier:,}")

print(f"\nTotal retained: {n_after_outlier:,} / {n0:,} ({n_after_outlier/n0:.1%})")

n = 4,508,106, k = 3.4143
Log-scale Q1: 3.3003, Q3: 5.0201, IQR: 1.7198
Log-scale bounds: [-2.5714, 10.8918]
2. before outlier removal: 4,508,106 rows
3. after log-N IQR outlier removal: 4,508,103 rows
Removed as outliers: 3

Total retained: 4,508,103 / 4,508,106 (100.0%)


In [23]:
# --- Outlier removal: dollar_value (log-N IQR rule) ---
# Raw dollar_value is right-skewed (many small purchases, a long tail of
# large ones), so IQR is computed on log1p(dollar_value) rather than the
# raw scale. Per the log-N rule: flag values more than k x IQR from the
# box, where k = sqrt(log n) - 0.5 for n > 100.

clean = clean.withColumn("log_dollar_value", F.log1p(F.col("dollar_value")))

n_rows = clean.count()
k = np.sqrt(np.log(n_rows)) - 0.5

q1, q3 = clean.approxQuantile("log_dollar_value", [0.25, 0.75], 0.0)
iqr = q3 - q1

lower_bound = q1 - k * iqr
upper_bound = q3 + k * iqr

print(f"n = {n_rows:,}, k = {k:.4f}")
print(f"Log-scale Q1: {q1:.4f}, Q3: {q3:.4f}, IQR: {iqr:.4f}")
print(f"Log-scale bounds: [{lower_bound:.4f}, {upper_bound:.4f}]")

n_before_outlier = report_shape(clean, "2. before outlier removal")

curated = clean.filter(
    (F.col("log_dollar_value") >= lower_bound) &
    (F.col("log_dollar_value") <= upper_bound)
).drop("log_dollar_value")

n_after_outlier = report_shape(curated, "3. after log-N IQR outlier removal")
print(f"Removed as outliers: {n_before_outlier - n_after_outlier:,}")

print(f"\nTotal retained: {n_after_outlier:,} / {n0:,} ({n_after_outlier/n0:.1%})")

n = 4,508,106, k = 3.4143
Log-scale Q1: 3.3003, Q3: 5.0201, IQR: 1.7198
Log-scale bounds: [-2.5714, 10.8918]


2. before outlier removal: 4,508,106 rows
3. after log-N IQR outlier removal: 4,508,103 rows
Removed as outliers: 3

Total retained: 4,508,103 / 4,508,106 (100.0%)
